In [ ]:
class Resource:
    def __init__(self, name, hourly_rate):
        self.name = name
        self.hourly_rate = hourly_rate
    def total_hours(self, projects):
        total_hours = 0
        for project in projects:
            if self in project.resources:
                index = project.resources.index(self)
                total_hours += project.hours[index]
        return total_hours

In [95]:
     
class Project:
    def __init__(self, client, name, project_type, start_date, end_date, resources, hours):
        self.name = name
        self.client = client
        self.project_type = project_type
        self.start_date = start_date
        self.end_date = end_date
        self.resources = resources
        self.hours = hours
        self.total_cost = sum([r.hourly_rate * h for r, h in zip(resources, hours)])
    
    def add_resource(self, resource, hours):
        self.resources.append(resource)
        self.hours.append(hours)
    
    def remove_resource(self, resource):
        index = self.resources.index(resource)
        self.resources.pop(index)
        self.hours.pop(index)
    
    def calculate_total_cost(self):
        self.total_cost = sum([r.hourly_rate * h for r, h in zip(self.resources, self.hours)])
        return self.total_cost
    
    def generate_report(self):
        report = "Project Report for {}:\n".format(self.name)
        report += "Client: {}\n".format(self.client)
        report += "Start Date: {}\n".format(self.start_date)
        report += "End Date: {}\n".format(self.end_date)
        report += "Resources:\n"
        for resource, hours in zip(self.resources, self.hours):
            report += "  {} - {} hours\n".format(resource.name, hours)
        report += "Total Cost: ${}\n".format(self.total_cost)
        return report
    
    @staticmethod
    def list_projects_for_client(client_name, projects):
        return [project for project in projects if project.client == client_name]
    
    @staticmethod
    def total_cost_of_all_projects(projects):
        return sum([project.total_cost for project in projects])

    """
    def get_monthly_hours(self):
        monthly_hours = {}
        date = self.start_date
        end_date = self.end_date
        while date < end_date:
            next_month = (date.month % 12) + 1
            next_year = date.year + (date.month + 1) // 12
            next_date = date.replace(month=next_month, year=next_year)
            days_in_month = (next_date - date).days
            total_days = (end_date - self.start_date).days
            for resource, hours in zip(self.resources, self.hours):
                resource_hours = hours * days_in_month / total_days
                monthly_hours[(resource, (date.year, date.month))] = int(resource_hours)
            date = next_date
    return monthly_hours
    """

    def get_monthly_hours(self):
        monthly_hours = {}
        date = self.start_date
        end_date = self.end_date
        total_hours = sum(self.hours)
        remaining_hours = total_hours
        while date < end_date:
            next_month = (date.month % 12) + 1
            next_year = date.year + (date.month + 1) // 12
            next_date = date.replace(month=next_month, year=next_year)
            days_in_month = (next_date - date).days
            total_days = (end_date - self.start_date).days
            for resource, hours in zip(self.resources, self.hours):
                resource_hours = hours * days_in_month / total_days
                if remaining_hours - resource_hours >= 0:
                    monthly_hours[(resource, (date.year, date.month))] = int(resource_hours)
                    remaining_hours -= resource_hours
                else:
                    monthly_hours[(resource, (date.year, date.month))] = int(remaining_hours)
                    break
            date = next_date

        return monthly_hours


In [98]:

def get_resource_hours_by_month(resource, projects):
    resource_hours = {}
    for project in projects:
        project_hours = project.get_monthly_hours()
        for (res, (year, month)), hours in project_hours.items():
            if res == resource:
                if (year, month) in resource_hours:
                    resource_hours[(year, month)] += hours
                else:
                    resource_hours[(year, month)] = hours
        return resource_hours

In [99]:
resources = [Resource("Pablo", 70), Resource("Fede", 45), Resource("Jeison", 70), Resource("Monica", 45), Resource("Sofia", 45)]


projects = [
    Project("Forbright", "CECL", "Validation", datetime(2023, 2, 1), datetime(2023, 4, 30), [resources[2], resources[3]], [105, 245]),
    Project("Forbright", "Verafin", "Validation", datetime(2023, 2, 15), datetime(2023, 5, 15), [resources[0], resources[1]], [120, 280]),
    Project("USBank", "Bonds", "Validation", datetime(2023, 3, 1), datetime(2023, 3, 31), [resources[0], resources[4]], [50, 100]),
    Project("USBank", "Bonds II", "Validation", datetime(2023, 3, 1), datetime(2023, 3, 31), [resources[0], resources[4]], [50, 100])]

In [100]:
project1 = projects[1]
project1.get_monthly_hours()




{(<__main__.Resource at 0x194e71bd4e0>, (2023, 2)): 37,
 (<__main__.Resource at 0x194e71bd840>, (2023, 2)): 88,
 (<__main__.Resource at 0x194e71bd4e0>, (2023, 3)): 41,
 (<__main__.Resource at 0x194e71bd840>, (2023, 3)): 97,
 (<__main__.Resource at 0x194e71bd4e0>, (2023, 4)): 40,
 (<__main__.Resource at 0x194e71bd840>, (2023, 4)): 94}

In [ ]:
resource = resources[0]


pablo_allocation = {}
for project in projects:
    for (resource, (year, month)), hours in project.get_monthly_hours().items():
        if resource == pablo:
            if (year, month) in pablo_allocation:
                pablo_allocation[(year, month)] += hours
            else:
                pablo_allocation[(year, month)] = hours

print("Pablo allocation:", pablo_allocation)